# OpenAI API

In [1]:
import openai
from openai import OpenAI

In [ ]:
client = OpenAI(api_key="")

In [6]:
client.chat.completions.create(
    model='gpt-3.5-turbo',
    messages=[
        {
            "role": "user",
            "content": "hello! who are you?"
        }
    ]
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# MinSearch

In [3]:
import minsearch
import json

In [4]:
def parse_docs():
    with open(r'../../01-intro/documents.json', 'rt') as f_in:
        docs_raw = json.load(f_in)
    #
    documents = []
    for course_dict in docs_raw:
        for doc in course_dict['documents']:
            doc['course'] = course_dict['course']
            documents.append(doc)
    #
    return documents

documents = parse_docs()
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [6]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"],
)
index.fit(documents)

In [7]:
def search(query, boost={'question': 3.0, 'section': 0.5},
           num_results=5, filter_dict={'course': 'data-engineering-zoomcamp'}):
    results = index.search(
        query=query,
        filter_dict=filter_dict,
        boost_dict=boost,
        num_results=num_results,
    )

    return results

def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

# Gemini API

In [8]:
from google import genai

In [ ]:
GEMINI_API_KEY=""
client = genai.Client(api_key=GEMINI_API_KEY)

def llm(prompt, model="gemini-2.0-flash", client=client):
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=prompt,
    )
    
    return response

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [11]:
query = 'how do I run kafka?'
response = rag(query)

In [13]:
print(response.text)

To run kafka with Java, navigate to the project directory, then run:
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java

To run kafka with Python, first create a virtual environment and run requirements.txt within that environment. To create the virtual environment and install packages (do this once), run:
python -m venv env
source env/bin/activate
pip install -r ../requirements.txt

To activate the virtual environment (do this each time you need it), run:
source env/bin/activate

To deactivate the virtual environment, run:
deactivate

Note that on Windows, the path is slightly different (it's env/Scripts/activate). Also, the virtual environment should be created only to run the python file, and Docker images should first all be up and running.

If you encounter a "ModuleNotFoundError: No module named 'kafka.vendor.six.moves'" error, use pip install kafka-python-ng instead.

